# Machine-learning trade model

This optional notebook uses lagged features, a purged chronological linear-regression workflow, and the shared backtester. Training labels whose 21-bar future return reaches the prediction date are purged. Prediction/target correlation remains descriptive because prediction dates are sparse and target windows overlap. The full trade ledger is saved as CSV; only a formatted 100-row preview is rendered.


In [ ]:
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next((path for path in candidates if (path / 'src' / 'trading').is_dir()), None)
if project_root is None:
    raise RuntimeError('Could not locate the parent project.')
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root / 'Machine Learning Trade Model' / 'src'))
from ml_trade_model.modeling import walk_forward_predictions
from trading.config import load_config
from trading.data import load_configured_market_data
from trading.risk import portfolio_risk_diagnostics
from trading.strategy import weekly_momentum_weights
from trading.backtest import format_performance_report, run_backtest
from trading.visualization import plot_diagnostics, plot_periodic_returns
config = load_config(project_root / 'config.yaml')
prices, universe_membership = load_configured_market_data(config, project_root)


In [ ]:
predictions, quality = walk_forward_predictions(prices, config['benchmark'], frequency=config['frequency'])
print('ML MODEL QUALITY')
print(f"Out-of-sample observations: {quality['observations']:,.0f}")
print(f"Prediction/realized-return correlation: {quality['correlation']:.3f}")
trade_asset_classes = set(config.values.get('trade_asset_classes', config.values.get('asset_classes', {})))
excluded_symbols = [symbol for asset_class, symbols in config.values.get('asset_classes', {}).items() if asset_class not in trade_asset_classes for symbol in symbols]
weights = weekly_momentum_weights(
    prices, config['benchmark'], config['top_n'], config['lookback_days'], config['skip_days'],
    config['volatility_days'], config['max_position_weight'], config['max_equity_exposure'],
    config['defensive_exposure'], config['stop_loss_pct'], config['drawdown_start'],
    config['drawdown_full_defense'], expected_returns=predictions,
    rebalance_frequency=config['frequency'],
    volatility_target=config['volatility_target'], excluded_symbols=excluded_symbols,
    benchmark_fallback=config.values.get('benchmark_fallback', False),
    universe_membership=universe_membership
)
ledger, holdings, trades = run_backtest(
    prices, weights, config['initial_capital'], config['commission_bps'],
    config['half_spread_bps'], config['slippage_bps'], config['market_impact_bps'],
    execution_lag_days=config['signal_execution_lag_days']
)
risk = portfolio_risk_diagnostics(holdings, prices, config['benchmark'])
print('ML RANKING MODEL (RESEARCH ONLY)')
print(format_performance_report(ledger, config['initial_capital']))
print(f'Trades recorded: {len(trades):,}')


In [ ]:
plot_diagnostics(ledger, trades, risk, prices[config['benchmark']])
plot_periodic_returns(ledger, period='Y')
plot_periodic_returns(ledger, period='M')
plot_periodic_returns(ledger, period='5Y')


In [ ]:
trade_path = project_root / 'data' / 'processed' / 'ml_trades.csv'
trades.to_csv(trade_path, index=False, date_format='%Y-%m-%d')
print(f'Full ML trade ledger saved to {trade_path}')
trade_preview = trades.head(100).copy()
trade_preview.style.format({
    'date': '{:%Y-%m-%d}', 'weight_before': '{:.2%}', 'weight_after': '{:.2%}',
    'weight_change': '{:.2%}', 'execution_price': '${:,.2f}', 'notional': '${:,.2f}',
    'estimated_cost': '${:,.2f}', 'realized_profit_loss': '${:,.2f}',
    'net_profit_loss': '${:,.2f}', 'profit_loss_pct': '{:.2%}',
    'cumulative_net_profit_loss': '${:,.2f}', 'cash_flow': '${:,.2f}',
    'cumulative_cash_flow': '${:,.2f}',
}, na_rep='—')
